In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import pyarrow.parquet as pq

sys.path.append(str(Path("..").resolve()))

from src.preprocessing.cleaning import clean_columns, replace_inf_with_nan, process_file_in_chunks

In [2]:
DATASET_PATH = Path("../data/raw/archive")
PROCESSED_PATH = Path("../data/processed")
MERGED_DIR = PROCESSED_PATH / "merged"
MERGED_FILE = MERGED_DIR / "ddos_merged.parquet"

# Only the cleaning cell below reads DATASET_PATH. Every later cell works off the
# Parquet output, so the rest of this notebook still runs after data/raw is removed.
csv_files = sorted(DATASET_PATH.rglob("*.csv"))
profile_df = pd.read_csv("../reports/dataset_summary.csv")
profile_df

,File,Rows,Columns,Missing Values,Duplicate Rows,Size (MB)
0,01-12/DrDoS_DNS.csv,5074413,88,31,0,2034.48
1,01-12/DrDoS_LDAP.csv,2181542,88,12,0,874.81
2,01-12/DrDoS_MSSQL.csv,4524498,88,14,0,1801.66
3,01-12/DrDoS_NetBIOS.csv,4094986,88,8,0,1618.84
4,01-12/DrDoS_NTP.csv,1217007,88,31,0,615.13
5,01-12/DrDoS_SNMP.csv,5161377,88,12,0,2071.93
6,01-12/DrDoS_SSDP.csv,2611374,88,2,0,1194.65
7,01-12/DrDoS_UDP.csv,3136802,88,8,0,1436.27
8,01-12/Syn.csv,1582681,88,202277,0,607.79
9,01-12/TFTP.csv,20107827,88,10497,0,8871.09


In [3]:
# Files where the full-dataset profiler found duplicate rows -- these get a
# full in-memory load + drop_duplicates() since each is small enough to fit
# safely in 24GB RAM on its own (see Size (MB) column above).
dedupe_files = set(profile_df.loc[profile_df["Duplicate Rows"] > 0, "File"])
dedupe_files

{'03-11/Syn.csv'}

In [4]:
for file in csv_files:
    rel_name = f"{file.parent.name}/{file.name}"
    dest_file = PROCESSED_PATH / file.parent.name / (file.stem + ".parquet")
    print(f"Cleaning {rel_name}...")
    process_file_in_chunks(
        file,
        dest_file,
        dedupe=rel_name in dedupe_files,
    )

Cleaning 01-12/DrDoS_DNS.csv...
Cleaning 01-12/DrDoS_LDAP.csv...
Cleaning 01-12/DrDoS_MSSQL.csv...
Cleaning 01-12/DrDoS_NetBIOS.csv...
Cleaning 01-12/DrDoS_NTP.csv...
Cleaning 01-12/DrDoS_SNMP.csv...
Cleaning 01-12/DrDoS_SSDP.csv...
Cleaning 01-12/DrDoS_UDP.csv...
Cleaning 01-12/Syn.csv...
Cleaning 01-12/TFTP.csv...
Cleaning 01-12/UDPLag.csv...
Cleaning 03-11/LDAP.csv...
Cleaning 03-11/MSSQL.csv...
Cleaning 03-11/NetBIOS.csv...
Cleaning 03-11/Portmap.csv...
Cleaning 03-11/Syn.csv...
Cleaning 03-11/UDP.csv...
Cleaning 03-11/UDPLag.csv...


In [5]:
check = pd.read_parquet(PROCESSED_PATH / "01-12" / "DrDoS_DNS.parquet")
check.head()

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,SimillarHTTP,Inbound,Label
0,172.16.0.5-192.168.50.1-634-60495-17,172.16.0.5,634.0,192.168.50.1,60495.0,17.0,2018-12-01 10:51:39.813448,28415.0,97.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,DrDoS_DNS
1,172.16.0.5-192.168.50.1-60495-634-17,192.168.50.1,634.0,172.16.0.5,60495.0,17.0,2018-12-01 10:51:39.820842,2.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,DrDoS_DNS
2,172.16.0.5-192.168.50.1-634-46391-17,172.16.0.5,634.0,192.168.50.1,46391.0,17.0,2018-12-01 10:51:39.852499,48549.0,200.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,DrDoS_DNS
3,172.16.0.5-192.168.50.1-634-11894-17,172.16.0.5,634.0,192.168.50.1,11894.0,17.0,2018-12-01 10:51:39.890213,48337.0,200.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,DrDoS_DNS
4,172.16.0.5-192.168.50.1-634-27878-17,172.16.0.5,634.0,192.168.50.1,27878.0,17.0,2018-12-01 10:51:39.941151,32026.0,200.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,DrDoS_DNS


In [6]:
parquet_files = sorted(p for p in PROCESSED_PATH.rglob("*.parquet") if p.parent != MERGED_DIR)

missing_report = []

for file in parquet_files:
    missing = 0
    rows = 0
    pf = pq.ParquetFile(file)
    for batch in pf.iter_batches(batch_size=500_000):
        chunk = batch.to_pandas()
        missing += int(chunk.isnull().sum().sum())
        rows += len(chunk)
    missing_report.append({
        "File": f"{file.parent.name}/{file.stem}",
        "Rows": rows,
        "Missing After Cleaning": missing,
        "Missing %": round(100 * missing / rows, 3),
    })

missing_df = pd.DataFrame(missing_report)
missing_df

,File,Rows,Missing After Cleaning,Missing %
0,01-12/DrDoS_DNS,5074413,324788,6.401
1,01-12/DrDoS_LDAP,2181542,77300,3.543
2,01-12/DrDoS_MSSQL,4524498,252932,5.590
3,01-12/DrDoS_NetBIOS,4094986,259706,6.342
4,01-12/DrDoS_NTP,1217007,14092,1.158
5,01-12/DrDoS_SNMP,5161377,21246,0.412
6,01-12/DrDoS_SSDP,2611374,84100,3.221
7,01-12/DrDoS_UDP,3136802,81346,2.593
8,01-12/Syn,1582681,404634,25.566
9,01-12/TFTP,20107827,1133522,5.637


In [7]:
missing_df.to_csv("../reports/missing_after_cleaning.csv", index=False)
missing_df

,File,Rows,Missing After Cleaning,Missing %
0,01-12/DrDoS_DNS,5074413,324788,6.401
1,01-12/DrDoS_LDAP,2181542,77300,3.543
2,01-12/DrDoS_MSSQL,4524498,252932,5.590
3,01-12/DrDoS_NetBIOS,4094986,259706,6.342
4,01-12/DrDoS_NTP,1217007,14092,1.158
5,01-12/DrDoS_SNMP,5161377,21246,0.412
6,01-12/DrDoS_SSDP,2611374,84100,3.221
7,01-12/DrDoS_UDP,3136802,81346,2.593
8,01-12/Syn,1582681,404634,25.566
9,01-12/TFTP,20107827,1133522,5.637


In [8]:
# Raw label values as they actually appear, per file. Filenames are NOT a reliable
# guide to contents -- e.g. 03-11/UDPLag is mostly Syn and UDP rows, and WebDDoS
# only exists inside 01-12/UDPLag. Labels must come from the Label column.
raw_label_counts = {}

for file in parquet_files:
    vc = pd.read_parquet(file, columns=["Label"])["Label"].value_counts()
    raw_label_counts[f"{file.parent.name}/{file.stem}"] = vc.to_dict()

raw_label_df = pd.DataFrame(raw_label_counts).fillna(0).astype(int)
raw_label_df

,01-12/DrDoS_DNS,01-12/DrDoS_LDAP,01-12/DrDoS_MSSQL,01-12/DrDoS_NetBIOS,01-12/DrDoS_NTP,01-12/DrDoS_SNMP,01-12/DrDoS_SSDP,01-12/DrDoS_UDP,01-12/Syn,01-12/TFTP,01-12/UDPLag,03-11/LDAP,03-11/MSSQL,03-11/NetBIOS,03-11/Portmap,03-11/Syn,03-11/UDP,03-11/UDPLag
DrDoS_DNS,5071011,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
BENIGN,3402,1612,2006,1707,14365,1507,763,2157,392,25247,3705,5124,2794,1321,4734,31386,3134,4068
DrDoS_LDAP,0,2179930,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
DrDoS_MSSQL,0,0,4522492,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
DrDoS_NetBIOS,0,0,0,4093279,0,0,0,0,0,0,0,0,0,0,0,0,0,0
DrDoS_NTP,0,0,0,0,1202642,0,0,0,0,0,0,0,0,0,0,0,0,0
DrDoS_SNMP,0,0,0,0,0,5159870,0,0,0,0,0,0,0,0,0,0,0,0
DrDoS_SSDP,0,0,0,0,0,0,2610611,0,0,0,0,0,0,0,0,0,0,0
DrDoS_UDP,0,0,0,0,0,0,0,3134645,0,0,0,0,0,0,0,0,0,0
Syn,0,0,0,0,0,0,0,0,1582289,0,0,0,0,0,0,3806356,0,606749


In [9]:
raw_label_df.to_csv("../reports/label_distribution.csv")

from src.preprocessing.labels import (
    LABEL_MAP,
    LABEL_ENCODING,
    DROPPED_LABELS,
    standardize_labels,
    save_encoding,
)

save_encoding(Path("../reports/label_encoding.json"))

print("dropped classes:", DROPPED_LABELS)
print("classes:", len(LABEL_ENCODING))
LABEL_ENCODING

dropped classes: {'WebDDoS'}
classes: 13


{'BENIGN': 0,
 'DNS': 1,
 'LDAP': 2,
 'MSSQL': 3,
 'NTP': 4,
 'NetBIOS': 5,
 'Portmap': 6,
 'SNMP': 7,
 'SSDP': 8,
 'Syn': 9,
 'TFTP': 10,
 'UDP': 11,
 'UDPLag': 12}

In [10]:
# Merge all files into one dataset, standardizing labels on the way through.
# Streamed batch-by-batch: the merged set is ~70M rows x 87 columns, which would be
# roughly 45GB as float64 in memory -- it never fits in RAM and must not be concatenated.
import pyarrow as pa
from collections import Counter

MERGED_DIR.mkdir(parents=True, exist_ok=True)

writer = None
class_totals = Counter()
rows_in = rows_out = 0

try:
    for file in parquet_files:
        print(f"Merging {file.parent.name}/{file.stem}...")
        for batch in pq.ParquetFile(file).iter_batches(batch_size=500_000):
            df = batch.to_pandas()
            rows_in += len(df)
            df = standardize_labels(df)
            rows_out += len(df)
            class_totals.update(df["Label"].value_counts().to_dict())

            table = pa.Table.from_pandas(df, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(MERGED_FILE, table.schema, compression="snappy")
            writer.write_table(table)
finally:
    if writer is not None:
        writer.close()

print(f"rows in: {rows_in:,}  rows out: {rows_out:,}  dropped: {rows_in - rows_out:,}")

Merging 01-12/DrDoS_DNS...
Merging 01-12/DrDoS_LDAP...
Merging 01-12/DrDoS_MSSQL...
Merging 01-12/DrDoS_NetBIOS...
Merging 01-12/DrDoS_NTP...
Merging 01-12/DrDoS_SNMP...
Merging 01-12/DrDoS_SSDP...
Merging 01-12/DrDoS_UDP...
Merging 01-12/Syn...
Merging 01-12/TFTP...
Merging 01-12/UDPLag...
Merging 03-11/LDAP...
Merging 03-11/MSSQL...
Merging 03-11/NetBIOS...
Merging 03-11/Portmap...
Merging 03-11/Syn...
Merging 03-11/UDP...
Merging 03-11/UDPLag...
rows in: 69,944,838  rows out: 69,944,399  dropped: 439


In [11]:
# Reconcile against the raw profiler counts: every row must be accounted for.
merged_rows = pq.ParquetFile(MERGED_FILE).metadata.num_rows
raw_rows = int(profile_df["Rows"].sum())
duplicates_removed = int(profile_df["Duplicate Rows"].sum())
expected = raw_rows - duplicates_removed - (rows_in - rows_out)

print(f"raw rows (profiler):   {raw_rows:,}")
print(f"duplicates removed:    {duplicates_removed:,}")
print(f"excluded classes:      {rows_in - rows_out:,}")
print(f"expected merged rows:  {expected:,}")
print(f"actual merged rows:    {merged_rows:,}")
print(f"reconciles: {merged_rows == expected}")

raw rows (profiler):   70,427,637
duplicates removed:    482,799
excluded classes:      439
expected merged rows:  69,944,399
actual merged rows:    69,944,399
reconciles: True


In [12]:
class_df = (
    pd.Series(class_totals, name="Rows")
    .sort_values(ascending=False)
    .rename_axis("Class")
    .reset_index()
)
class_df["Share %"] = (100 * class_df["Rows"] / class_df["Rows"].sum()).round(4)
class_df["Imbalance vs largest"] = (class_df["Rows"].max() / class_df["Rows"]).round(1)

class_df.to_csv("../reports/class_distribution.csv", index=False)
class_df

,Class,Rows,Share %,Imbalance vs largest
0,TFTP,20082580,28.7122,1.0
1,MSSQL,10309945,14.7402,1.9
2,NetBIOS,7750776,11.0813,2.6
3,UDP,7001800,10.0105,2.9
4,Syn,5995394,8.5717,3.3
5,SNMP,5159870,7.3771,3.9
6,DNS,5071011,7.2501,4.0
7,LDAP,4095052,5.8547,4.9
8,SSDP,2610611,3.7324,7.7
9,NTP,1202642,1.7194,16.7
